In [ ]:
import numpy as np
import torch
from torchvision import transforms
import tensorflow as tf

In [5]:
# Mounting our drive onto the colab notebook
from google.colab import drive
drive.mount('/content/drive')
# Importing the convolutional network class
# !pip install import-ipynb
# import import_ipynb
# import CNNTensorFlowLayersAPI.ipynb
#-------------Helper functions to reshape the dataset-------------#

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
def loadDataNp(path):
  """Helper function to import the datasets and their respective labels."""
  maps = np.load(path,allow_pickle=True)
  return maps

def reshapeSample(sample):
  """Helper function to take the flattened pixels that form a sample and
  reshape them into the original 2d occupancy map."""
  # We know beforehand the the original occupancy maps were
  # [0,100] x [0,100] maps
  sampleReshaped = np.reshape(sample,(100,100))
  # breakpoint()
  return sampleReshaped

def prepareData(dataset):
  # We will iterate as many samples as there are in the dataset
  datasetDimensions = np.shape(dataset)
  # Total number of samples
  breakpoint()
  totalSamples = datasetDimensions[0]
  breakpoint()
  # Total pixels
  #totalPixels = datasetDimensions[1]
  # Numpy array to store reshaped samples
  preparedSamplesArray = np.empty(totalSamples,dtype='O')
  breakpoint()

  for index in range(totalSamples):# Offset to access last sample inclusive
    # Grabbing an entire row since this is the flattened out 2d occupancy map
    prepSample = reshapeSample(dataset[index,:])
    # breakpoint()
    # Storing the current prepared sample
    preparedSamplesArray[index] = prepSample


    # breakpoint()

  return preparedSamplesArray

#-----------------------------------------IMPORTING MNIST BENCHMARK TEST DATSET-------------------------------------#
"""
breakpoint()
mnistPath = '/content/drive/MyDrive/OccupancyMapsDataset/mnist_scaled.npz'
# Importing the MNIST dataset
mnist = np.load(mnistPath)
#breakpoint()
trainingSet = mnist['trainingSet']
#breakpoint()
trainingSetLabels = mnist['trainingLabels']
#breakpoint()
testingSet = mnist['testingSet']
#breakpoint()
testingSetLables = mnist['testingLabels']
#breakpoint()
"""



"""IMPORTING THE TRAINING DATA AND TRAINING LABELS. BOTH ARE IN THE FORMAT OF
COMPRESSED NUMPY ARRAYS CREATED USING THE NUMPY.SAVEZ() FUNCTION."""

#----------------------------------------------TRAINING DATASET------------------------------------------------------#
# Getting the path for the training dataset
pathTrainingSet = '/content/drive/MyDrive/OccupancyMapsDataset/trainingSet.npz'
# Importing the compressed numpy array
# trainingDataNP = loadDataNp(pathTrainingSet)
trainingDataNP = loadDataNp(pathTrainingSet)
# TODO: Adding breakpoint to inspect state of importing compressed numpy array
# breakpoint()
# Accessing the actual training data
trainingSet = trainingDataNP['trainingSet']
# TODO: Adding breakpoint to inspect the value of loadedTrainingMaps
# breakpoint()
# Gets just a single occupancy map
loadedFirstMap = trainingSet[2]
# TODO: Adding a breakpoint to assess the value of a single map
# breakpoint()
#loadedTrainingMapsReshaped = prepareData(trainingSet)
# TODO: Adding a breakpoint to see whether we have successfully reshaped dataset
breakpoint()


#----------------------------------------------TRAINING DATASET LABELS---------------------------------------------------#
pathTrainingSetLabels = '/content/drive/MyDrive/OccupancyMapsDataset/trainingSetLabels.npz'
# Importing the compresses numpy array with training set labels
trainingDataLabelsNP = loadDataNp(pathTrainingSetLabels)
# TODO: Adding breakpoint to inspect state of importing compressed numpy array
breakpoint()
# Accessing the actual training data labels
trainingSetLabels = trainingDataLabelsNP['trainingSetLables']
# TODO: Adding breakpoint to inspect the value of loadedTrainingMapsLabels
breakpoint()


#-------------Helper functions to reshape the dataset-------------#
"""
def loadDataNp(path):
  #Helper function to import the datasets and their respective labels.
  maps = np.load(path,allow_pickle=True)
  return maps

def reshapeSample(sample):
  # Helper function to take the flattened pixels that form a sample and
  # reshape them into the original 2d occupancy map.
  # We know beforehand the the original occupancy maps were
  # [0,100] x [0,100] maps
  sampleReshaped = np.reshape(sample,(28,28))
  # breakpoint()
  return sampleReshaped

def prepareData(dataset):
  # We will iterate as many samples as there are in the dataset
  datasetDimensions = np.shape(dataset)
  # Total number of samples
  breakpoint()
  totalSamples = datasetDimensions[0]
  breakpoint()
  # Total pixels
  #totalPixels = datasetDimensions[1]
  # Numpy array to store reshaped samples
  preparedSamplesArray = np.empty(totalSamples,dtype='O')
  breakpoint()

  for index in range(totalSamples):# Offset to access last sample inclusive
    # Grabbing an entire row since this is the flattened out 2d occupancy map
    prepSample = reshapeSample(dataset[index,:])
    # breakpoint()
    # Storing the current prepared sample
    preparedSamplesArray[index] = prepSample


    # breakpoint()

  return preparedSamplesArray


"""
# We need to create a dataloader so that we can feed the samples to the convolutional neural network
class YourDataset(torch.utils.data.Dataset):
  def __init__(self,numpyArrayDataset,numpyArrayLabels):
    self.dataset = numpyArrayDataset
    self.labels = numpyArrayLabels

  def __getitem__(self,index):
    #This method will return to us transformed ssamples and labels
    # Grabbing the sample
    data,label = self.dataset[index], self.labels[index]
    # Creating transform
    transform = transforms.Compose([
        transforms.ToTensor()# Dataset is already normalized
    ])
    # Returning the data
    # Using transpose to convert the data into NHWC format from NCHW format
    transformedData = transform(data)
    transformedData = torch.permute(transformedData,(1,2,0))
    # return transform(data), torch.tensor(label)
    return transformedData, torch.tensor(label)
  def __len__(self):
    return len(self.dataset)


# Transforming the data
reshapedDataset = prepareData(trainingSet[5000:60000])
reshapedValidation = prepareData(trainingSet[:5000])
#breakpoint()

BATCH_SIZE = 64
# Calculating the steps per epoch
steps_per_epoch = np.ceil(40000/BATCH_SIZE)
# COnverting steps per epoch into an integer
steps_per_epoch = int(steps_per_epoch)
train_set = YourDataset(reshapedDataset,trainingSetLabels[5000:60000])
validation_set = YourDataset(reshapedValidation,trainingSetLabels[:5000])
breakpoint()
# Using the dataloader
train_dataloader = torch.utils.data.DataLoader(train_set,
                                               batch_size = BATCH_SIZE,
                                               num_workers = 1,
                                               shuffle=True)
validation_dataloader = torch.utils.data.DataLoader(validation_set,
                                                    batch_size=64)
breakpoint()
# Iterating over the dataLoader
counter = 1
# Reshaping the tensors into NHWC format
"""
for samples, labels in train_dataloader:
  samples = tf.transpose(samples,perm=[0,2,3,1])
  print(labels.shape)
  print(counter)
  counter += 1
breakpoint()
for samples, labels in validation_dataloader:
  samples = tf.transpose(samples,perm=[0,2,3,1])
  print(labels.shape)
  print(counter)
  counter += 1
"""
for samples, labels in train_dataloader:
  print(samples.shape)
  print(counter)
  counter += 1
breakpoint()
#------------------BUILDING THE CONVOLUTIONAL NEURAL NETWORK-----------------#
"""
model = tf.keras.Sequential([tf.keras.layers.Conv2D(32,(3,3),padding='same',activation='relu',data_format='channels_first'),
                             tf.keras.layers.MaxPooling2D((2,2),data_format='channels_first'),
                             tf.keras.layers.Dropout(rate=0.5),

                             tf.keras.layers.Conv2D(
                                 64,(3,3), padding='same',activation='relu',data_format='channels_first'),
                             tf.keras.layers.MaxPooling2D((2,2),data_format='channels_first'),
                             tf.keras.layers.Dropout(rate=0.5),

                             tf.keras.layers.Conv2D(128,(3,3),padding='same',activation='relu',data_format='channels_first'),
                             tf.keras.layers.MaxPooling2D((2,2),data_format='channels_first'),

                             tf.keras.layers.Conv2D(256,(3,3),padding='same',activation='relu',data_format='channels_first'),

                            ])
# Data format is NHWC
model.compute_output_shape(input_shape = (None,1,28,28))
breakpoint()
# model.add(tf.keras.layers.GlobalAveragePooling2D())
model.add(tf.keras.layers.Flatten(data_format='channels_first'))
# Data format is NHWC
model.compute_output_shape(input_shape=(None,1,28,28))
breakpoint()
model.add(tf.keras.layers.Dense(10,activation='softmax'))
# Radom seed
tf.random.set_seed(1)
# Building the model
# Data format is NHWC
model.build(input_shape=(None,1,28,28))
model.summary()
# Compiling the model
"""
"""
model.compile(optimizer=tf.keras.optimizers.Adam(),loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics = ['accuracy'])
"""
#-----------------------------MNIST DATSET SPECIFIC-------------------------------#
model = tf.keras.Sequential()
model.add(tf.keras.layers.Conv2D(filters=32,kernel_size=(5,5),strides=(1,1),
                                 padding='same',data_format='channels_last',name='conv_1',activation='relu'))
model.add(tf.keras.layers.AveragePooling2D(pool_size=(2,2),name='pool_1',data_format='channels_last'))
#model.add(tf.keras.layers.Dropout(rate=0.5))
model.add(tf.keras.layers.Conv2D(filters=64,kernel_size=(5,5),strides=(1,1),
                                 padding='same',name='conv_2',activation='relu',data_format='channels_last'))
model.add(tf.keras.layers.AveragePooling2D(pool_size=(2,2),name='pool_2',data_format='channels_last'))
# model.add(tf.keras.layers.Dropout(rate=0.5))
model.add(tf.keras.layers.Conv2D(filters=128,kernel_size=(5,5),strides=(1,1),
                                 padding='same',name='conv_3',activation='relu',data_format='channels_last'))
model.add(tf.keras.layers.AveragePooling2D(pool_size=(2,2),name='pool_3',data_format='channels_last'))
#---------REMOVING TO CHECK THE PERFORMANCE WITH LOW COMPLEXITY
# model.add(tf.keras.layers.Dropout(rate=0.5))
model.add(tf.keras.layers.Conv2D(filters=256,kernel_size=(5,5),strides=(1,1),
                                 padding='same',name='conv_4',activation='relu',data_format='channels_last'))
model.add(tf.keras.layers.AveragePooling2D(pool_size=(2,2),name='pool_4',data_format='channels_last'))
# model.add(tf.keras.layers.Dropout(rate=0.5))
"""
model.add(tf.keras.layers.Conv2D(filters=512,kernel_size=(5,5),strides=(1,1),
                                 padding='same',name='conv_5',activation='relu',data_format='channels_last'))
model.add(tf.keras.layers.MaxPool2D(pool_size=(2,2),name='pool_5',data_format='channels_last'))
"""
#------------------------------------------------#
'''
#-----------Added at 60 percent accuracy.
model.add(tf.keras.layers.Conv2D(filters=1024,kernel_size=(5,5),strides=(1,1),
                                 padding='same',name='conv_6',activation='relu',data_format='channels_last'))
model.add(tf.keras.layers.MaxPool2D(pool_size=(2,2),name='pool_6',data_format='channels_last'))
#---------------------------------------#.
'''
model.add(tf.keras.layers.Flatten(data_format='channels_last'))
model.add(tf.keras.layers.Dense(units=1024,name='fc_1',activation='relu'))
# Changing loss function to SparseCategoricalCrossentropy
model.add(tf.keras.layers.Dropout(rate=0.5))
model.add(tf.keras.layers.Dense(units=7,name='fc_2',activation='softmax'))
# Building the model
tf.random.set_seed(1)
model.build(input_shape=(None,100,100,1))
model.summary()
breakpoint()
# Compiling the model
# Changing loss function to SparseCategoricalCrossentropy
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics = ['accuracy'])
# Calling the model on the validation and training datasets
# Removing steps per epoch
# history = model.fit(train_dataloader,validation_data = validation_dataloader,epochs=20,steps_per_epoch = steps_per_epoch)
breakpoint()
history = model.fit(train_dataloader,validation_data = validation_dataloader,epochs=20,shuffle=True)

--Return--
None
> <ipython-input-6-5c77dd8905b4>(80)<cell line: 0>()
     78 #loadedTrainingMapsReshaped = prepareData(trainingSet)
     79 # TODO: Adding a breakpoint to see whether we have successfully reshaped dataset
---> 80 breakpoint()
     81 
     82 

ipdb> continue
--Return--
None
> <ipython-input-6-5c77dd8905b4>(88)<cell line: 0>()
     86 trainingDataLabelsNP = loadDataNp(pathTrainingSetLabels)
     87 # TODO: Adding breakpoint to inspect state of importing compressed numpy array
---> 88 breakpoint()
     89 # Accessing the actual training data labels
     90 trainingSetLabels = trainingDataLabelsNP['trainingSetLables']

ipdb> continue
--Return--
None
> <ipython-input-6-5c77dd8905b4>(92)<cell line: 0>()
     90 trainingSetLabels = trainingDataLabelsNP['trainingSetLables']
     91 # TODO: Adding breakpoint to inspect the value of loadedTrainingMapsLabels
---> 92 breakpoint()
     93 
     94 

ipdb> continue
> <ipython-input-6-5c77dd8905b4>(20)prepareData()
     18   # Total

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_1 (Conv2D)                 │ (None, 100, 100, 32)   │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (AveragePooling2D)       │ (None, 50, 50, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_2 (Conv2D)                 │ (None, 50, 50, 64)     │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (AveragePooling2D)       │ (None, 25, 25, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_3 (Conv2D)                 │ (None, 25, 25, 128)    │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_3 (AveragePooling2D)       │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_4 (Conv2D)                 │ (None, 12, 12, 256)    │       819,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_4 (AveragePooling2D)       │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc_1 (Dense)                    │ (None, 1024)           │     9,438,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc_2 (Dense)                    │ (None, 7)              │         7,175 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,521,863 (40.14 MB)

 Trainable params: 10,521,863 (40.14 MB)

 Non-trainable params: 0 (0.00 B)

--Return--
None
> <ipython-input-6-5c77dd8905b4>(283)<cell line: 0>()
    281 model.build(input_shape=(None,100,100,1))
    282 model.summary()
--> 283 breakpoint()
    284 # Compiling the model
    285 # Changing loss function to SparseCategoricalCrossentropy

ipdb> continue
--Return--
None
> <ipython-input-6-5c77dd8905b4>(291)<cell line: 0>()
    288 # Calling the model on the validation and training datasets
    289 # Removing steps per epoch
    290 # history = model.fit(train_dataloader,validation_data = validation_dataloader,epochs=20,steps_per_epoch = steps_per_epoch)
--> 291 breakpoint()
    292 history = model.fit(train_dataloader,validation_data = validation_dataloader,epochs=20,shuffle=True)

ipdb> continue
Epoch 1/20
860/860 ━━━━━━━━━━━━━━━━━━━━ 28s 28ms/step - accuracy: 0.5009 - loss: 1.0285 - val_accuracy: 0.6046 - val_loss: 0.8507
Epoch 2/20
860/860 ━━━━━━━━━━━━━━━━━━━━ 22s 25ms/step - accuracy: 0.6170 - loss: 0.8427 - val_accuracy: 0.6400 - val_loss: 0.8033
Epoch 3/20
8

In [ ]:
import tensorflow

In [ ]:
a = "tyring"
a

'tyring'